# 04 - Count-based Representations (Part A)
**Owner:** Asjad Aleem

Bag of Words, TF-IDF, N-grams on our cleaned Urdu news corpus (`data/processed/cleaned.csv` from Phase 1). Code: `src/count_vectors.py`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
df = pd.read_csv(ROOT / "data/processed/cleaned.csv").dropna(subset=["clean_text"])
docs, labels = df["clean_text"].tolist(), df["label"].tolist()
print(len(docs), "documents")

In [ ]:
from count_vectors import build_bow, build_tfidf, ngram_growth, sparsity

## A1. Bag of Words
A document-term matrix: one row per article, one column per word, each cell = how many times the word appears.

In [ ]:
bow_vec, X_bow = build_bow(docs)
print("BoW shape   :", X_bow.shape, "(documents x vocabulary)")
print("Non-zero    :", f"{X_bow.nnz:,}")
print("Sparsity    :", f"{sparsity(X_bow):.4f}% of entries are zero")
print("Avg distinct words per doc:", round(X_bow.nnz / X_bow.shape[0], 1))

**Comment (edit in your own words):** almost every cell is zero, because one article uses only a few hundred of the 100k+ words in the vocabulary.

## A2. TF-IDF
TF-IDF = how often a word is in THIS document x how rare it is across ALL documents. sklearn default: `idf = ln((1+N)/(1+df)) + 1`, then each row is L2-normalized.

In [ ]:
tfidf_vec, X_tfidf = build_tfidf(docs, min_df=2)
print("TF-IDF shape:", X_tfidf.shape, " sparsity:", f"{sparsity(X_tfidf):.4f}%")
vocab = np.array(tfidf_vec.get_feature_names_out())
idf = tfidf_vec.idf_
order = np.argsort(idf)
print("\nLowest IDF (in almost every document):")
print(pd.DataFrame({"word": vocab[order[:10]], "idf": idf[order[:10]].round(3)}).to_string(index=False))

### Concrete example: one article

In [ ]:
i = 0   # change to look at another article
row = X_tfidf[i].toarray()[0]
bow_row = bow_vec.transform([docs[i]]).toarray()[0]
nz = np.nonzero(row)[0]
tab = pd.DataFrame({"word": vocab[nz], "tfidf": row[nz].round(4)})
tab["count_in_doc"] = [bow_row[bow_vec.vocabulary_[w]] for w in tab["word"]]
tab["idf"] = [idf[tfidf_vec.vocabulary_[w]].round(3) for w in tab["word"]]
tab = tab.sort_values("tfidf", ascending=False)
print("Category:", labels[i]); print("Text:", docs[i][:200], "...")
print("\nHIGHEST weights:"); print(tab.head(8).to_string(index=False))
print("\nLOWEST weights (driven toward zero):"); print(tab.tail(8).to_string(index=False))

**Why (edit in your own words):** words at the top are frequent in this article but rare in the corpus (high idf), so they describe what THIS article is about.
Words at the bottom (e.g. `NUM` and very common news words) appear in thousands of articles, so their idf is close to 1 and, after normalization, their weight is pushed toward zero even if they appear many times.

## A3. N-grams: how the vocabulary grows
On the first 5,000 articles (trigrams on the full corpus need too much RAM).

In [ ]:
sub = docs[:5000]
g1 = ngram_growth(sub, min_df=1); g5 = ngram_growth(sub, min_df=5)
print("All n-grams (min_df=1):"); print(g1.to_string(index=False))
print("\nOnly n-grams in >= 5 docs (min_df=5):"); print(g5.to_string(index=False))

In [ ]:
ax = g1.plot(x="n-grams", y="features", kind="bar", legend=False, color="#4C72B0", title="Features vs n-gram range (5,000 docs)")
ax.set_ylabel("number of features"); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(ROOT / "figures/ngram_growth.png", dpi=150); plt.show()

**Comment (edit in your own words):** adding bigrams and trigrams multiplies the number of features, but most new n-grams appear only once, so the matrix gets even more sparse.
With `min_df=5` the growth is much smaller: most bigrams/trigrams are rare noise, while a few (like names `سری لنکا`, `ورلڈ کپ`) are useful phrases.

## A4. Hand-worked numerical (toy example)
3 tiny documents. We compute ONE TF-IDF weight and ONE bigram by hand, then check with sklearn.
The hand working (on paper, CamScanned) is attached in the report.

| Doc | Text |
|---|---|
| d1 | پاکستان کرکٹ جیت |
| d2 | پاکستان معیشت ترقی |
| d3 | کرکٹ میچ جیت |

In [ ]:
toy = ["پاکستان کرکٹ جیت", "پاکستان معیشت ترقی", "کرکٹ میچ جیت"]
N = len(toy)

# ---- TF-IDF of "معیشت" in d2, by hand ----
tf = 1                                  # appears once in d2
df_t = 1                                # appears in 1 of 3 documents
idf_m = np.log((1 + N) / (1 + df_t)) + 1
print(f"idf(معیشت)  = ln((1+{N})/(1+{df_t})) + 1 = ln(2) + 1 = {idf_m:.4f}")
idf_p = np.log((1 + N) / (1 + 2)) + 1   # پاکستان is in 2 docs
idf_t = idf_m                           # ترقی is also in 1 doc
print(f"idf(پاکستان) = ln(4/3) + 1 = {idf_p:.4f}")
raw = {"پاکستان": 1 * idf_p, "معیشت": 1 * idf_m, "ترقی": 1 * idf_t}
norm = np.sqrt(sum(v**2 for v in raw.values()))
print("raw tf*idf in d2:", {k: round(v, 4) for k, v in raw.items()})
print(f"L2 norm of d2   = sqrt({idf_p:.4f}^2 + {idf_m:.4f}^2 + {idf_t:.4f}^2) = {norm:.4f}")
hand = raw["معیشت"] / norm
print(f"TF-IDF(معیشت, d2) by hand = {raw['معیشت']:.4f} / {norm:.4f} = {hand:.4f}")

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
tv = TfidfVectorizer(token_pattern=r"\S+", lowercase=False)
Xt = tv.fit_transform(toy)
lib = Xt[1, tv.vocabulary_["معیشت"]]
print(f"TF-IDF(معیشت, d2) sklearn = {lib:.4f}   match: {np.isclose(hand, lib)}")

In [ ]:
# ---- bigram, by hand ----
print("d1 tokens :", toy[0].split())
print("d1 bigrams by hand: (پاکستان کرکٹ), (کرکٹ جیت)")
print("count of 'کرکٹ جیت' in d1 by hand = 1, in d3 = 0 (d3 has 'کرکٹ میچ' and 'میچ جیت')")
cv = CountVectorizer(token_pattern=r"\S+", lowercase=False, ngram_range=(2, 2))
Xb = cv.fit_transform(toy)
print("\nsklearn bigram vocabulary:", list(cv.get_feature_names_out()))
j = cv.vocabulary_["کرکٹ جیت"]
print("sklearn count of 'کرکٹ جیت' -> d1:", Xb[0, j], " d3:", Xb[2, j])